# FP-QSIM Bell State Demo

This notebook shows a minimal Bell-state example and compares the custom simulator against a Qiskit reference statevector.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

from fp_qsim.simulator import CustomSimulatorManual
from fp_qsim.state_vector import mocked_statevector

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

custom = CustomSimulatorManual()
ref_sim = AerSimulator(method="statevector")

circuit_ucx = transpile(qc, basis_gates=["u", "cx"])
compiled_circuit = transpile(circuit_ucx, ref_sim)

custom_statevector = custom.run(circuit_ucx, shots=1024)
reference_statevector = mocked_statevector(compiled_circuit)

In [ ]:
anchor = int(np.argmax(np.abs(custom_statevector)))
phase = reference_statevector[anchor] / custom_statevector[anchor]
aligned_custom = custom_statevector * phase

print("Reference statevector:", reference_statevector)
print("Custom statevector (phase aligned):", aligned_custom)
print("Match:", np.allclose(reference_statevector, aligned_custom))

In [ ]:
basis_labels = ["|00>", "|01>", "|10>", "|11>"]
probs_ref = np.abs(reference_statevector) ** 2
probs_custom = np.abs(aligned_custom) ** 2

x = np.arange(len(basis_labels))
width = 0.35

plt.figure(figsize=(8, 4))
plt.bar(x - width / 2, probs_ref, width, label="Reference")
plt.bar(x + width / 2, probs_custom, width, label="Custom", alpha=0.8)
plt.xticks(x, basis_labels)
plt.ylim(0, 0.6)
plt.ylabel("Probability")
plt.title("Bell state probabilities")
plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.show()